# Elliptic Curve Cryptography: From Theory to Bitcoin

**Source material:** *Elliptic Curve Cryptography research paper* (695.744 Reverse Engineering & Vulnerability Analysis)  
**Related:** [The Wright Trick (2016)](./01-ecc-wright-trick.ipynb) | [Nonsense Signature (2018)](./02-ecc-nonsense-signature.ipynb) | [Original Paper Notebook](./03-ecc-original-paper.ipynb) | [Projective Sphere](./04-ecc-projective-sphere.ipynb)

---

## Start at the End: What Problem Does ECC Solve?

How do you **prove you wrote something** — a message, a document, an authorization —
to someone who has never met you, across an untrusted network, without revealing
your secret?

That's the **digital signature problem.** In Bitcoin the stakes are concrete:
the ledger is public — every transaction is visible to everyone. When you want
to move bitcoin, you broadcast an entry that says *"move X from address A to
address B"* and the entire network can read it. So how do you prove you control
address A, without exposing the thing that gives you control? You can't show a
password — everyone would see it. You can't rely on a trusted third party —
there isn't one. You need a proof that only you could produce, that anyone can
verify, that reveals nothing about your secret, and that can't be replayed.

That's a digital signature — and that's what ECC gives us. Before ECC, we already had answers:
RSA (1977) and ElGamal (1985) both work. But they need enormous keys — 3072 bits
for 128-bit security. That's slow, wasteful, and impractical for constrained devices.

In 1985, Neal Koblitz and Victor Miller independently asked: **what if we ran the
same discrete logarithm trick, but on a different mathematical structure — one where
the problem is fundamentally harder?**

The structure they found was **elliptic curves over finite fields.** Same security,
a fraction of the key size:

| Security | RSA / ElGamal | ECC | Savings |
|----------|--------------|-----|---------|
| 80-bit   | 1024 bits    | 160 bits | 6x smaller |
| 128-bit  | 3072 bits    | 256 bits | 12x smaller |
| 256-bit  | 15360 bits   | 512 bits | 30x smaller |

### What we need from the math

To build a digital signature scheme, we need exactly three things:

1. **A one-way function** — easy to compute forward, infeasible to reverse
   - ECC answer: $P = d \times G$ (scalar multiplication on a curve)
   - Forward: milliseconds. Reverse: more energy than the solar system contains.
   - **In Bitcoin this fires twice:** once when you create a wallet ($d \to P$ becomes your public key), and once every time you sign a transaction (a fresh random $k \to R = k \times G$ becomes part of the signature).

2. **A signing operation** — only the secret holder can produce it
   - ECDSA answer: $s = k^{-1}(z + r \cdot d) \bmod N$
   - Requires the private key $d$. No other way to produce a valid $(r, s)$.

3. **A public verification** — anyone can check it without learning the secret
   - ECDSA answer: check if $R'_x = r$ where $R' = (z/s)G + (r/s)P$
   - Uses only the public key $P$, the message hash $z$, and the signature $(r, s)$.

### But WHY does this work?

It works because elliptic curves over finite fields give us a one-way function
where "multiplication" means something geometrically elegant — drawing lines
through curve points, finding intersections, reflecting. And because the curve
points form an **algebraic group**, all the familiar rules of arithmetic hold,
but reversing the operation (the **discrete logarithm**) is computationally
infeasible.

To build this one-way function, we need a very specific mathematical environment —
one where numbers wrap around, where parallel lines meet at infinity, and where
points on a curve obey group axioms.

**That's the journey of this notebook:** starting from what we need to accomplish,
we build the mathematical machinery piece by piece, and see it come alive in
working code.

---

## How to Read This

Each module answers a question that the previous module raised:

```
"We need a digital signature scheme with small keys"
  └─→ "We need a one-way function on a curve"
        └─→ Module 4: The Discrete Log Problem — what makes it hard?
              └─→ Module 3: Point Operations — how do we "add" and "multiply"?
                    └─→ Module 2: Elliptic Curves — what are these curves?
                          └─→ Module 1: Algebraic Foundations — what rules govern this world?

Then we build back up with the answer:
  Module 5: ECDSA — the signature scheme we set out to build
  Module 6: Bitcoin Applications — where it all lands (Schnorr, onion routing)
  Module 7: Exercises — test your understanding
```

If you prefer, jump straight to Module 3 (the code) and circle back to
Modules 1-2 (the theory) when you want to understand WHY the formulas work.



# Module 1: Algebraic Foundations

We need a one-way function: a secret $d$, a public starting point $G$, and an
operation where $P = d \times G$ is trivial to compute but impossible to reverse.
The signer uses $d$ to produce a signature; the verifier checks it against $P$
and the message without ever learning $d$.

But the operation $\times$ can't just be anything. For signing and verification
to work — for the equations to be rearrangeable, for inverses to exist, for both
sides to arrive at the same answer — **the math has to behave well.** Specifically:

| Constraint | Why | Mathematical name |
|-----------|-----|-------------------|
| Combining two elements always produces another valid element | Otherwise the system breaks after one operation | **Closure** |
| Grouping doesn't matter: $(a \cdot b) \cdot c = a \cdot (b \cdot c)$ | Otherwise the signing equation wouldn't simplify | **Associativity** |
| There's a neutral "do nothing" element | We need a concept of "zero" / "identity point" | **Identity** |
| Every operation can be undone | Verification requires computing inverses | **Inverse** |
| Order doesn't matter: $a \cdot b = b \cdot a$ | Signer and verifier must arrive at the same result independently | **Commutativity** |

A system satisfying the first four is a **group**. Add commutativity and it's
an **Abelian group**. This is exactly the structure that points on an elliptic
curve form — and it's why elliptic curves are the foundation of Bitcoin's
signature scheme.

---

## 1.1 Groups — the formal definition

A **group** is a set $S$ with a binary operation $\cdot$ satisfying four axioms:

| Axiom | Statement | Intuition |
|-------|-----------|----------|
| **Closure** | $\forall a,b \in S: a \cdot b \in S$ | Combining elements stays in the set |
| **Associativity** | $(a \cdot b) \cdot c = a \cdot (b \cdot c)$ | Grouping doesn't matter |
| **Identity** | $\exists e: a \cdot e = e \cdot a = a$ | There's a "do nothing" element |
| **Inverse** | $\forall a, \exists a^{-1}: a \cdot a^{-1} = e$ | Every action can be undone |

An **Abelian group** adds a fifth property:

| **Commutativity** | $a \cdot b = b \cdot a$ | Order doesn't matter |

Now you can see why we need all five. The signing equation in ECDSA is
$s = k^{-1}(z + r \cdot d) \bmod N$. Verification rearranges it to
$R' = (z/s)G + (r/s)P$. That rearrangement **only works** because the
underlying group is Abelian — you can commute, associate, and invert freely.

Without closure, $P = d \times G$ might not even be a valid point.
Without inverses, verification couldn't compute $s^{-1}$.
Without commutativity, the signer and verifier would get different answers.

In [1]:
# Demonstration: integers mod n form an Abelian group under addition

n = 7
S = list(range(n))  # {0, 1, 2, 3, 4, 5, 6}

# Closure: adding any two elements stays in the set
print("=== Group Axioms for Z_7 under addition ===")
set_str = "{" + ", ".join(map(str, S)) + "}"
print(f"\nClosure: (3 + 5) mod 7 = {(3 + 5) % n}  ✓ (still in {set_str})")

# Associativity
a, b, c = 2, 4, 6
lhs = ((a + b) % n + c) % n
rhs = (a + (b + c) % n) % n
print(f"Associativity: ({a}+{b})+{c} = {lhs},  {a}+({b}+{c}) = {rhs}  ✓")

# Identity: 0 is the identity
print(f"Identity: 5 + 0 = {(5 + 0) % n}  ✓")

# Inverse: every element has one
for x in S:
    inv = (n - x) % n
    assert (x + inv) % n == 0
print(f"Inverses: every element has an inverse  ✓")
print(f"  Example: inverse of 3 is {(n - 3) % n} because (3 + {(n-3)%n}) mod 7 = {(3 + (n-3)) % n}")

# Commutativity (Abelian)
print(f"Commutativity: 2 + 5 = {(2+5)%n},  5 + 2 = {(5+2)%n}  ✓  (Abelian)")

=== Group Axioms for Z_7 under addition ===

Closure: (3 + 5) mod 7 = 1  ✓ (still in {0, 1, 2, 3, 4, 5, 6})
Associativity: (2+4)+6 = 5,  2+(4+6) = 5  ✓
Identity: 5 + 0 = 5  ✓
Inverses: every element has an inverse  ✓
  Example: inverse of 3 is 4 because (3 + 4) mod 7 = 0
Commutativity: 2 + 5 = 0,  5 + 2 = 0  ✓  (Abelian)


## 1.2 Rings and Fields

A **ring** is a set with two operations (addition and multiplication) where:
- Addition forms an Abelian group
- Multiplication is associative and distributes over addition

A **field** is a commutative ring where every nonzero element has a multiplicative inverse.

### The crucial field for Bitcoin: $\mathbb{F}_p$

Given a prime $p$, the set $\{0, 1, 2, \ldots, p-1\}$ with modular arithmetic
forms a **finite field** (also called a Galois field).

Every nonzero element has a multiplicative inverse because $p$ is prime.

In [2]:
# Finite field F_11: every nonzero element has a multiplicative inverse

p = 11
print(f"=== Finite Field F_{p} ===")
print(f"Elements: {{0, 1, ..., {p-1}}}")
print(f"\nMultiplicative inverses (a × a⁻¹ ≡ 1 mod {p}):")

for a in range(1, p):
    # Fermat's little theorem: a^(p-2) mod p = a^(-1) mod p
    inv = pow(a, p - 2, p)
    print(f"  {a:2d}⁻¹ = {inv:2d}   (verify: {a} × {inv} = {a * inv} ≡ {(a * inv) % p} mod {p})")

=== Finite Field F_11 ===
Elements: {0, 1, ..., 10}

Multiplicative inverses (a × a⁻¹ ≡ 1 mod 11):
   1⁻¹ =  1   (verify: 1 × 1 = 1 ≡ 1 mod 11)
   2⁻¹ =  6   (verify: 2 × 6 = 12 ≡ 1 mod 11)
   3⁻¹ =  4   (verify: 3 × 4 = 12 ≡ 1 mod 11)
   4⁻¹ =  3   (verify: 4 × 3 = 12 ≡ 1 mod 11)
   5⁻¹ =  9   (verify: 5 × 9 = 45 ≡ 1 mod 11)
   6⁻¹ =  2   (verify: 6 × 2 = 12 ≡ 1 mod 11)
   7⁻¹ =  8   (verify: 7 × 8 = 56 ≡ 1 mod 11)
   8⁻¹ =  7   (verify: 8 × 7 = 56 ≡ 1 mod 11)
   9⁻¹ =  5   (verify: 9 × 5 = 45 ≡ 1 mod 11)
  10⁻¹ = 10   (verify: 10 × 10 = 100 ≡ 1 mod 11)


## 1.3 Equivalence Classes and Modular Arithmetic

An **equivalence relation** $R$ on a set $S$ satisfies:
- **Reflexivity:** $a R a$
- **Symmetry:** $a R b \Rightarrow b R a$
- **Transitivity:** $a R b$ and $b R c \Rightarrow a R c$

**Congruence modulo $n$** is the equivalence relation that partitions integers
into $n$ classes:

$$a \equiv b \pmod{n} \iff n \mid (a - b)$$

This is the mechanism that makes finite fields work — we "wrap around" at $p$,
creating a closed system where arithmetic never escapes the set.

In [3]:
# Equivalence classes modulo 5

n = 5
print(f"=== Equivalence Classes mod {n} ===")
for cls in range(n):
    members = [x for x in range(-10, 21) if x % n == cls]
    print(f"  [{cls}] = {{ {', '.join(map(str, members))} ... }}")

print(f"\nAll of these are 'the same' in mod {n}:")
print(f"  -7 ≡ -2 ≡ 3 ≡ 8 ≡ 13 (mod 5)")
print(f"  Verify: all give remainder {3 % 5} when divided by 5")

=== Equivalence Classes mod 5 ===
  [0] = { -10, -5, 0, 5, 10, 15, 20 ... }
  [1] = { -9, -4, 1, 6, 11, 16 ... }
  [2] = { -8, -3, 2, 7, 12, 17 ... }
  [3] = { -7, -2, 3, 8, 13, 18 ... }
  [4] = { -6, -1, 4, 9, 14, 19 ... }

All of these are 'the same' in mod 5:
  -7 ≡ -2 ≡ 3 ≡ 8 ≡ 13 (mod 5)
  Verify: all give remainder 3 when divided by 5


## 1.4 Projective Space

Standard (affine) coordinates break down at the **point at infinity** — a concept
we need for elliptic curves to form a proper group.

**Projective space** $\mathbb{P}^n$ extends affine space $\mathbb{A}^n$ by adding
"points at infinity" where parallel lines meet.

A point in projective space is an equivalence class of triples:

$$[X : Y : Z] \sim [\lambda X : \lambda Y : \lambda Z] \quad \text{for } \lambda \neq 0$$

### Affine ↔ Projective Conversion

- Affine $(x, y)$ → Projective $[x : y : 1]$
- Projective $[X : Y : Z]$ → Affine $(X/Z, Y/Z)$ when $Z \neq 0$
- **Point at infinity:** $[0 : 1 : 0]$ (where $Z = 0$)

The elliptic curve $y^2 = x^3 + ax + b$ in projective form becomes:

$$Y^2 Z = X^3 + aXZ^2 + bZ^3$$

Setting $Z = 0$: $0 = X^3$, so $X = 0$ and the point at infinity is $[0 : 1 : 0]$.

In [4]:
# Stereographic projection: mapping a circle to a line
# This illustrates how projective space "wraps" a line with a point at infinity

import math

def stereographic_project(x, z):
    """Project point (x,z) on unit circle from north pole (0,1) to x-axis."""
    if z == 1:  # North pole maps to infinity
        return float('inf')
    return x / (1 - z)

def stereographic_inverse(r):
    """Map point r on x-axis back to unit circle."""
    if r == float('inf'):
        return (0, 1)  # North pole
    x = 2 * r / (r**2 + 1)
    z = (r**2 - 1) / (r**2 + 1)
    return (x, z)

print("=== Stereographic Projection (Circle → Line) ===")
print("Point on circle  →  Point on line")
print("-" * 42)

angles = [0, math.pi/6, math.pi/4, math.pi/3, math.pi/2, 
          2*math.pi/3, 5*math.pi/6, math.pi]

for theta in angles:
    x = math.sin(theta)
    z = -math.cos(theta)  # South pole at bottom
    r = stereographic_project(x, z)
    if r == float('inf'):
        print(f"  ({x:6.3f}, {z:6.3f})  →  ∞  (point at infinity!)")
    else:
        print(f"  ({x:6.3f}, {z:6.3f})  →  {r:6.3f}")

print(f"\nThe line ℝ plus the point at infinity = projective line P¹")
print(f"This is how elliptic curves 'close' themselves into a group.")

=== Stereographic Projection (Circle → Line) ===
Point on circle  →  Point on line
------------------------------------------
  ( 0.000, -1.000)  →   0.000
  ( 0.500, -0.866)  →   0.268
  ( 0.707, -0.707)  →   0.414
  ( 0.866, -0.500)  →   0.577
  ( 1.000, -0.000)  →   1.000
  ( 0.866,  0.500)  →   1.732
  ( 0.500,  0.866)  →   3.732
  ( 0.000,  1.000)  →  ∞  (point at infinity!)

The line ℝ plus the point at infinity = projective line P¹
This is how elliptic curves 'close' themselves into a group.


### Visualizing the Projective Plane

The text demo above shows the *idea* — now let's **see** it.

We take the Bitcoin curve $y^2 = x^3 + 7$ in the ordinary (affine) plane and lift it
onto a unit sphere using **inverse stereographic projection** from the north pole:

$$
(u, v) \;\longmapsto\;
\left(
  \frac{2u}{u^2+v^2+1},\;
  \frac{2v}{u^2+v^2+1},\;
  \frac{u^2+v^2-1}{u^2+v^2+1}
\right)
$$

In the affine plane the two branches of the curve diverge to $\pm\infty$.
On the sphere **they close up and meet at the north pole** — that single point
*is* the point at infinity $\mathcal{O}$ that makes the curve a group.

*(Inspired by [Trustica's video](https://www.youtube.com/watch?v=Hk0Fr-k7wmQ)
on the Weierstrass curve in the projective plane.)*

In [5]:
import numpy as np
import plotly.graph_objects as go

def inv_stereo(u, v):
    """Inverse stereographic projection: plane (u,v) → unit sphere (X,Y,Z).
    North pole (0,0,1) is the projection centre; (0,0) maps to south pole."""
    d = u**2 + v**2 + 1
    return 2*u/d, 2*v/d, (u**2 + v**2 - 1)/d

# ── grid ────────────────────────────────────────────────────────────────
span = 2 * np.pi
n_grid = 30                               # grid density (lines, not fill)
n_pts  = 200                              # points per grid line (smoothness)
vals = np.linspace(-span, span, n_grid)   # where the grid lines sit
line_t = np.linspace(-span, span, n_pts)  # parametric samples along each line

# Also build a dense fill mesh for surface colouring
n_fill = 80
fill_v = np.linspace(-span, span, n_fill)
Xf, Yf = np.meshgrid(fill_v, fill_v)
Xs_f, Ys_f, Zs_f = inv_stereo(Xf, Yf)

# Pre-build the grid lines as flat arrays with None breaks (one Scatter3d each)
def build_grid_lines(map_fn):
    """Return (gx, gy, gz) for all grid lines, with None separators."""
    gx, gy, gz = [], [], []
    for v in vals:                         # horizontal lines (constant y = v)
        xs, ys, zs = map_fn(line_t, np.full_like(line_t, v))
        gx += list(xs) + [None]
        gy += list(ys) + [None]
        gz += list(zs) + [None]
    for v in vals:                         # vertical lines   (constant x = v)
        xs, ys, zs = map_fn(np.full_like(line_t, v), line_t)
        gx += list(xs) + [None]
        gy += list(ys) + [None]
        gz += list(zs) + [None]
    return gx, gy, gz

def flat_map(u, v):
    return u, v, np.zeros_like(u)

grid_flat = build_grid_lines(flat_map)
grid_sphere = build_grid_lines(inv_stereo)

# ── Bitcoin curve  y² = x³ + 7  (single closed loop) ───────────────────
x_min = -(7 ** (1/3))
t_near = np.linspace(x_min + 1e-8, 5, 1500)
t_far  = np.geomspace(5, 800, 1500)       # geometric spacing → denser near 5
t_all = np.concatenate([t_near, t_far])
t_all = t_all[t_all**3 + 7 >= 0]
y_all = np.sqrt(t_all**3 + 7)

# Trace: upper branch forward, then lower branch in reverse → closed loop
loop_u = np.concatenate([t_all, t_all[::-1]])
loop_v = np.concatenate([y_all, -y_all[::-1]])

# On the sphere (lifted slightly so it sits above the surface)
_lx, _ly, _lz = inv_stereo(loop_u, loop_v)
R_LIFT = 1.012
curve_sx = R_LIFT * np.array(_lx)
curve_sy = R_LIFT * np.array(_ly)
curve_sz = R_LIFT * np.array(_lz)

# Clipped for the flat-plane view
mask = (t_all <= span) & (y_all <= span)
t_f = t_all[mask]
yp_f, yn_f = y_all[mask], -y_all[mask]
flat_loop_u = np.concatenate([t_f, t_f[::-1]])
flat_loop_v = np.concatenate([yp_f, yn_f[::-1]])

GRID_COLOR = "rgba(0,0,0,0.45)"
GRID_W = 1
CURVE_W = 5

# ── y² = x³ − x  (two components — egg + branch) ───────────────────────
t_egg = np.linspace(-1 + 1e-8, -1e-8, 800)
y_egg = np.sqrt(t_egg**3 - t_egg)
egg_u = np.concatenate([t_egg, t_egg[::-1]])
egg_v = np.concatenate([y_egg, -y_egg[::-1]])
_ex, _ey, _ez = inv_stereo(egg_u, egg_v)

t_br_near = np.linspace(1 + 1e-8, 5, 1000)
t_br_far  = np.geomspace(5, 600, 1200)
t_br = np.concatenate([t_br_near, t_br_far])
y_br = np.sqrt(t_br**3 - t_br)
br_u = np.concatenate([t_br, t_br[::-1]])
br_v = np.concatenate([y_br, -y_br[::-1]])
_bx, _by, _bz = inv_stereo(br_u, br_v)

R_LIFT = 1.012
egg_sx, egg_sy, egg_sz = R_LIFT*np.array(_ex), R_LIFT*np.array(_ey), R_LIFT*np.array(_ez)
br_sx, br_sy, br_sz    = R_LIFT*np.array(_bx), R_LIFT*np.array(_by), R_LIFT*np.array(_bz)

# ── y² = x³ − x + 0.5  (ONE component — single big loop around sphere) ─
a_w, b_w = -1, 0.5
roots_w = np.roots([1, 0, a_w, b_w])
x_min_w = roots_w[np.abs(roots_w.imag) < 1e-10].real.min()
t_w_near = np.linspace(x_min_w + 1e-8, 5, 2000)
t_w_far  = np.geomspace(5, 800, 1500)
t_w = np.concatenate([t_w_near, t_w_far])
rhs_w = t_w**3 + a_w*t_w + b_w
t_w = t_w[rhs_w >= 0]
y_w = np.sqrt(t_w**3 + a_w*t_w + b_w)
wrap_u = np.concatenate([t_w, t_w[::-1]])
wrap_v = np.concatenate([y_w, -y_w[::-1]])
_wx, _wy, _wz = inv_stereo(wrap_u, wrap_v)
wrap_sx = R_LIFT * np.array(_wx)
wrap_sy = R_LIFT * np.array(_wy)
wrap_sz = R_LIFT * np.array(_wz)

# ── y² = x³ + 7  (secp256k1, single component) ────────────────────────
x_min_btc = -(7 ** (1/3))
t_btc_near = np.linspace(x_min_btc + 1e-8, 5, 1500)
t_btc_far  = np.geomspace(5, 800, 1500)
t_btc = np.concatenate([t_btc_near, t_btc_far])
t_btc = t_btc[t_btc**3 + 7 >= 0]
y_btc = np.sqrt(t_btc**3 + 7)
btc_u = np.concatenate([t_btc, t_btc[::-1]])
btc_v = np.concatenate([y_btc, -y_btc[::-1]])
_cx, _cy, _cz = inv_stereo(btc_u, btc_v)
btc_sx, btc_sy, btc_sz = R_LIFT*np.array(_cx), R_LIFT*np.array(_cy), R_LIFT*np.array(_cz)

# flat-plane clipped versions
mask_btc = (t_btc <= span) & (y_btc <= span)
t_bf, yp_bf = t_btc[mask_btc], y_btc[mask_btc]
flat_btc_u = np.concatenate([t_bf, t_bf[::-1]])
flat_btc_v = np.concatenate([yp_bf, -yp_bf[::-1]])

print("Shared data ready — run the next two cells.")

Shared data ready — run the next two cells.


In [6]:
# ═══════════════════════════════════════════════════════════════════════
#  AFFINE PLANE — flat grid  +  y² = x³ + 7
# ═══════════════════════════════════════════════════════════════════════
fig_a = go.Figure()

# coloured fill (no contours — we draw the real grid lines ourselves)
fig_a.add_trace(go.Surface(
    x=Xf, y=Yf, z=np.zeros_like(Xf),
    surfacecolor=Zs_f,
    colorscale="Viridis", showscale=False, opacity=0.80,
))

# parametric grid lines
gx, gy, gz = grid_flat
fig_a.add_trace(go.Scatter3d(
    x=gx, y=gy, z=gz,
    mode="lines", line=dict(color=GRID_COLOR, width=GRID_W),
    showlegend=False, hoverinfo="skip",
))

# secp256k1:  y² = x³ + 7  (red)
fig_a.add_trace(go.Scatter3d(
    x=flat_btc_u, y=flat_btc_v, z=np.full(len(flat_btc_u), 0.12),
    mode="lines", line=dict(color="rgb(255,34,0)", width=CURVE_W),
    name="y² = x³ + 7", showlegend=True,
))

# y² = x³ − x  egg (cyan) — clipped to visible range
fig_a.add_trace(go.Scatter3d(
    x=egg_u, y=egg_v, z=np.full(len(egg_u), 0.12),
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x (egg)", showlegend=True,
))
# y² = x³ − x  branch (cyan) — clipped
mask_br_flat = (t_br <= span) & (y_br <= span)
t_brf, y_brf = t_br[mask_br_flat], y_br[mask_br_flat]
fig_a.add_trace(go.Scatter3d(
    x=np.concatenate([t_brf, t_brf[::-1]]),
    y=np.concatenate([y_brf, -y_brf[::-1]]),
    z=np.full(2*len(t_brf), 0.12),
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x (branch)", showlegend=True,
))

fig_a.update_layout(
    title="Affine plane  —  two curves on the grid",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="",
        camera=dict(eye=dict(x=0.4, y=-1.1, z=1.7)),
        zaxis=dict(range=[-0.5, 0.5], showticklabels=False),
        aspectmode="manual", aspectratio=dict(x=1.2, y=1.2, z=0.10),
    ),
    height=650, width=950,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_a.show()
print("Uniform grid squares. The colour shows where each square lands on the sphere (next cell).")

Uniform grid squares. The colour shows where each square lands on the sphere (next cell).


In [7]:
# ═══════════════════════════════════════════════════════════════════════
#  PROJECTIVE SPHERE — same grid warped via stereographic projection
#  Drag to rotate, or hit ▶ Spin.
# ═══════════════════════════════════════════════════════════════════════
fig_s = go.Figure()

# coloured fill (no built-in contours)
fig_s.add_trace(go.Surface(
    x=Xs_f, y=Ys_f, z=Zs_f,
    surfacecolor=Zs_f,
    colorscale="Viridis", showscale=False, opacity=0.65,
))

# real warped grid lines (each straight line on the plane becomes a curve here)
gx, gy, gz = grid_sphere
fig_s.add_trace(go.Scatter3d(
    x=gx, y=gy, z=gz,
    mode="lines", line=dict(color=GRID_COLOR, width=GRID_W),
    showlegend=False, hoverinfo="skip",
))

# secp256k1: y² = x³ + 7  (red — hugs upper hemisphere)
fig_s.add_trace(go.Scatter3d(
    x=btc_sx, y=btc_sy, z=btc_sz,
    mode="lines", line=dict(color="rgb(255,34,0)", width=CURVE_W),
    name="y² = x³ + 7  (secp256k1)", showlegend=True,
))

# y² = x³ − x  egg (cyan — loops through lower hemisphere)
fig_s.add_trace(go.Scatter3d(
    x=egg_sx, y=egg_sy, z=egg_sz,
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x  (egg)", showlegend=True,
))

# y² = x³ − x  branch (cyan — sweeps up to north pole)
fig_s.add_trace(go.Scatter3d(
    x=br_sx, y=br_sy, z=br_sz,
    mode="lines", line=dict(color="rgb(0,200,255)", width=CURVE_W),
    name="y² = x³ − x  (branch)", showlegend=True,
))

# y² = x³ − x + 0.5  (white — single big loop wrapping the whole sphere)
fig_s.add_trace(go.Scatter3d(
    x=wrap_sx, y=wrap_sy, z=wrap_sz,
    mode="lines", line=dict(color="white", width=CURVE_W + 1),
    name="y² = x³ − x + ½  (single loop)", showlegend=True,
))

# point at infinity
fig_s.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[1.03],
    mode="markers+text",
    marker=dict(size=8, color="white", line=dict(color="red", width=3)),
    text=["  𝒪 (point at ∞)"], textposition="top right",
    textfont=dict(size=14, color="red", family="serif"),
    showlegend=False,
))

fig_s.update_layout(
    title="Projective sphere  —  drag to rotate  /  ▶ Spin",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        camera=dict(eye=dict(x=1.4, y=-1.4, z=0.6)),
        aspectmode="data",
    ),
    height=700, width=950,
    margin=dict(l=0, r=0, t=40, b=0),
)

# auto-spin
n_frames = 72
frames = []
for i in range(n_frames):
    a = 2 * np.pi * i / n_frames
    frames.append(go.Frame(
        layout=dict(scene=dict(camera=dict(
            eye=dict(x=1.8*np.cos(a), y=1.8*np.sin(a), z=0.6)))),
        name=str(i),
    ))
fig_s.frames = frames
fig_s.update_layout(updatemenus=[dict(
    type="buttons", showactive=False,
    x=0.95, y=0.02, xanchor="right", yanchor="bottom",
    buttons=[
        dict(label="▶ Spin", method="animate",
             args=[None, dict(frame=dict(duration=80, redraw=True),
                              fromcurrent=True, mode="immediate",
                              transition=dict(duration=0))]),
        dict(label="⏸ Stop", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False),
                                mode="immediate")]),
    ],
)])

fig_s.show()
print("Red   = y² = x³ + 7      (secp256k1)  — small loop hugging the north pole")
print("Cyan  = y² = x³ − x      (2 components — egg + branch, two separate circles)")
print("White = y² = x³ − x + ½  (1 component — single loop sweeping around the sphere)")
print("\nToggle each in the legend. The white curve is what the Trustica video shows.")

Red   = y² = x³ + 7      (secp256k1)  — small loop hugging the north pole
Cyan  = y² = x³ − x      (2 components — egg + branch, two separate circles)
White = y² = x³ − x + ½  (1 component — single loop sweeping around the sphere)

Toggle each in the legend. The white curve is what the Trustica video shows.


### Key Takeaway

Projective space gives us:
1. A well-defined **point at infinity** $\mathcal{O} = [0:1:0]$
2. This point serves as the **identity element** for the group
3. Every line through two curve points intersects the curve at exactly one more point
4. No more edge cases from "parallel lines" — they meet at infinity

---

# Module 2: Elliptic Curves

## 2.1 The Curve Equation

An elliptic curve over a field $\mathbb{F}$ is defined by the **Weierstrass equation**:

$$y^2 = x^3 + ax + b$$

with the constraint that $4a^3 + 27b^2 \neq 0$ (non-singular — no cusps or self-intersections).

### Bitcoin's curve: secp256k1

$$y^2 = x^3 + 7 \pmod{P}$$

where $a = 0$, $b = 7$ (a **Koblitz curve** — the zero $a$ enables optimizations).

In [8]:
# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

=== secp256k1 Parameters ===
Curve: y² = x³ + 0x + 7
P = 2²⁵⁶ - 2³² - 977
  = 115792089237316195423570985008687907853269984665640564039457584007908834671663
  (256 bits)

N = 115792089237316195423570985008687907852837564279074904382605163141518161494337
  (256 bits)

Non-singular check: 4(0)³ + 27(7)² = 1323 ≠ 0  ✓

P mod 4 = 3  (enables efficient square roots)


## 2.2 Elliptic Curves Over Finite Fields

Over the real numbers, an elliptic curve is a smooth curve. Over a **finite field** $\mathbb{F}_p$,
it becomes a discrete set of points — there are exactly $N$ of them (plus the point at infinity).

Let's visualize a small elliptic curve to build intuition before working with secp256k1's
enormous numbers.

In [9]:
# Small curve: y² = x³ + x + 1 over F_23
# (Using a=1, b=1, p=23 for a visible example)

p_small = 23
a_small, b_small = 1, 1

def is_quadratic_residue(n, p):
    """Is n a perfect square mod p? (Euler's criterion)"""
    if n % p == 0:
        return True
    return pow(n, (p - 1) // 2, p) == 1

def mod_sqrt_small(a, p):
    """Square root mod p for p ≡ 3 (mod 4)."""
    return pow(a, (p + 1) // 4, p)

points = []
for x in range(p_small):
    rhs = (x**3 + a_small * x + b_small) % p_small
    if is_quadratic_residue(rhs, p_small):
        y = mod_sqrt_small(rhs, p_small)
        points.append((x, y))
        if y != 0 and y != p_small - y:  # Two y values unless y=0
            points.append((x, p_small - y))
        elif y == 0:
            pass  # Only one point when y=0

points.sort()
print(f"=== Curve: y² = x³ + {a_small}x + {b_small} over F_{p_small} ===")
print(f"Number of points: {len(points)} (+ point at infinity = {len(points) + 1})")
print(f"\nAll points:")
for i, (x, y) in enumerate(points):
    end = '\n' if (i + 1) % 4 == 0 else '   '
    print(f"  ({x:2d}, {y:2d})", end=end)
print()

# ASCII visualization
print(f"\n{'─' * 50}")
print(f"Visual (x across, y up):")
grid = [['·' for _ in range(p_small)] for _ in range(p_small)]
for x, y in points:
    grid[p_small - 1 - y][x] = '■'

for row_idx, row in enumerate(grid):
    y_val = p_small - 1 - row_idx
    if y_val % 4 == 0:
        print(f"{y_val:2d} |{''.join(row)}")
print(f"   +{'─' * p_small}")
print(f"    ", end='')
for x in range(p_small):
    if x % 4 == 0:
        print(f"{x}", end=' ' * (4 - len(str(x))))
print()

=== Curve: y² = x³ + 1x + 1 over F_23 ===
Number of points: 27 (+ point at infinity = 28)

All points:
  ( 0,  1)     ( 0, 22)     ( 1,  7)     ( 1, 16)
  ( 3, 10)     ( 3, 13)     ( 4,  0)     ( 5,  4)
  ( 5, 19)     ( 6,  4)     ( 6, 19)     ( 7, 11)
  ( 7, 12)     ( 9,  7)     ( 9, 16)     (11,  3)
  (11, 20)     (12,  4)     (12, 19)     (13,  7)
  (13, 16)     (17,  3)     (17, 20)     (18,  3)
  (18, 20)     (19,  5)     (19, 18)   

──────────────────────────────────────────────────
Visual (x across, y up):
20 |···········■·····■■····
16 |·■·······■···■·········
12 |·······■···············
 8 |·······················
 4 |·····■■·····■··········
 0 |····■··················
   +───────────────────────
    0   4   8   12  16  20  


### Observation

Notice the **vertical symmetry** — for every point $(x, y)$ there's a point $(x, p-y)$.
This is because if $y^2 \equiv c \pmod{p}$, then $(-y)^2 \equiv c \pmod{p}$ too.
The negation of a point is its vertical mirror: $-P = (x, -y)$.

---

# Module 3: Point Operations

This is where the algebraic structure from Module 1 meets the curves from Module 2.
We define an "addition" operation on curve points that satisfies all the group axioms.

## 3.1 Geometric Intuition

```
Point Addition P + Q:          Point Doubling 2P:

     ·  Q                           ·
    / \                            /|\
   /   \                          / | \
  P     \  ← secant line        P  | tangent line
   \     \                        \ |
    \     T                        \T
     \   /                          |
      \ /                           |
       R = P + Q  (reflect T)       R = 2P  (reflect T)
```

1. Draw a line through P and Q (or the tangent at P for doubling)
2. The line hits the curve at a third point T
3. Reflect T across the x-axis to get R = P + Q

## 3.2 The Formulas

In [10]:
from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

Generator G:     (0x79be667ef9dcbb..., 0x483ada7726a3c4...)
Identity O:      O (point at infinity)


In [11]:
def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

G on curve? y² mod P == x³+7 mod P: True  ✓

2G = (0xc6047f9441ed7d..., 0x1ae168fea63dc3...)
G + O = G? True  ✓  (identity)
G + (-G) = O? True  ✓  (inverse)


## 3.3 Scalar Multiplication (Double-and-Add)

**The most important operation in ECC.** Computing $k \times G$ gives us
a public key from a private key.

Naive approach: add G to itself $k$ times → $O(k)$ operations.  
**Double-and-add**: use binary representation of $k$ → $O(\log k)$ operations.

```
Example: 13 × P  (13 = 1101 in binary)

Step  Binary  Action             Result
─────────────────────────────────────────
  0   1       result += addend    P
      ─       addend = 2×addend   2P
  1   0       (skip add)          P
      ─       addend = 2×addend   4P
  2   1       result += addend    P + 4P = 5P
      ─       addend = 2×addend   8P
  3   1       result += addend    5P + 8P = 13P
```

In [12]:
def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

3G (double-and-add): (0xf9308a019258c3..., 0x388f7b0f632de8...)
3G (G + 2G):         (0xf9308a019258c3..., 0x388f7b0f632de8...)
Match: True  ✓

Fundamental: N × G = O (point at infinity)
This means private key space is cyclic with order N.
N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.


## 3.4 Compressed Public Keys

Since $y^2 = x^3 + 7$ has at most two solutions for $y$ given any $x$,
we only need to store $x$ plus one bit indicating which $y$ (even or odd).

```
Uncompressed: 65 bytes  [04 || x (32 bytes) || y (32 bytes)]
Compressed:   33 bytes  [02/03 || x (32 bytes)]
                         02 = even y,  03 = odd y
```

In [13]:
def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")

=== Key Pair Generation ===
Private key (d):  0x16f32fb4cd7c1578ea...
Public key (P = d×G):
  x: 0x99f87c1479a457947a894d5cbf307a68cfa6b2e67b5fc2d7501f84b7330531c7
  y: 0x1ed309201e9c42893c57d6ad2356122ffa24648139bdb4e8867fc655c459bd7b
Compressed: 0399f87c1479a457947a894d5cbf307a68cfa6b2e67b5fc2d7501f84b7330531c7
  Prefix 0x03 → y is odd

Round-trip: True  ✓


---

# Module 4: From ElGamal to ECC

The essay draws a parallel between ElGamal and ECC. Both rely on the hardness
of the **discrete logarithm problem**, but in different mathematical settings.

## 4.1 The Discrete Logarithm Problem

| Setting | Easy Direction | Hard Direction |
|---------|---------------|----------------|
| **ElGamal** (integers mod p) | $\beta = \alpha^a \bmod p$ | Given $\beta, \alpha, p$, find $a$ |
| **ECC** (elliptic curve) | $Q = k \times P$ | Given $Q, P$, find $k$ |

Both are "one-way": trivial to compute forward, infeasible to reverse.

**But ECC wins on efficiency.** A 256-bit ECC key provides the same security as a
3072-bit RSA/ElGamal key.

| Security Level | RSA/ElGamal Key | ECC Key | Ratio |
|---------------|----------------|---------|-------|
| 80-bit | 1024 bits | 160 bits | 6.4× |
| 128-bit | 3072 bits | 256 bits | 12× |
| 256-bit | 15360 bits | 512 bits | 30× |

In [14]:
# Side-by-side: ElGamal vs ECC key exchange

print("=" * 60)
print("ElGamal Key Exchange (integers mod p)")
print("=" * 60)

# Small ElGamal example (from the essay)
p_eg = 101
alpha = 7  # Primitive root mod 101

# Alice
a_priv = 23  # Alice's private key
beta_a = pow(alpha, a_priv, p_eg)  # Alice's public key
print(f"Alice: private a={a_priv}, public β = α^a mod p = 7^{a_priv} mod 101 = {beta_a}")

# Bob
b_priv = 37  # Bob's private key
beta_b = pow(alpha, b_priv, p_eg)  # Bob's public key
print(f"Bob:   private b={b_priv}, public β'= α^b mod p = 7^{b_priv} mod 101 = {beta_b}")

# Shared secret
shared_eg_a = pow(beta_b, a_priv, p_eg)  # Alice computes β'^a
shared_eg_b = pow(beta_a, b_priv, p_eg)  # Bob computes β^b
print(f"Shared secret: Alice={shared_eg_a}, Bob={shared_eg_b}, Match={shared_eg_a == shared_eg_b}")
print(f"(Both compute α^(ab) mod p = 7^{a_priv*b_priv} mod 101 = {pow(alpha, a_priv*b_priv, p_eg)})")

print(f"\n{'=' * 60}")
print("ECC Key Exchange (ECDH on secp256k1)")
print("=" * 60)

# Alice
alice_priv = secrets.randbelow(SECP_N - 1) + 1
alice_pub = scalar_mult(alice_priv, G)  # Alice's public key = a × G
print(f"Alice: private a (256-bit random), public A = a×G")

# Bob
bob_priv = secrets.randbelow(SECP_N - 1) + 1
bob_pub = scalar_mult(bob_priv, G)  # Bob's public key = b × G
print(f"Bob:   private b (256-bit random), public B = b×G")

# Shared secret: both compute the same point
shared_ecc_a = scalar_mult(alice_priv, bob_pub)   # a × (b×G) = ab×G
shared_ecc_b = scalar_mult(bob_priv, alice_pub)    # b × (a×G) = ab×G
print(f"Shared secret point: {shared_ecc_a == shared_ecc_b}  ✓")
print(f"Both compute a×b×G (same point, never transmitted)")

print(f"\n{'─' * 60}")
print(f"ElGamal: security from α^a mod p  (needs ~3072-bit p)")
print(f"ECC:     security from k×G         (needs ~256-bit N)")
print(f"Same security, 12× smaller keys.")

ElGamal Key Exchange (integers mod p)
Alice: private a=23, public β = α^a mod p = 7^23 mod 101 = 27
Bob:   private b=37, public β'= α^b mod p = 7^37 mod 101 = 35
Shared secret: Alice=94, Bob=94, Match=True
(Both compute α^(ab) mod p = 7^851 mod 101 = 94)

ECC Key Exchange (ECDH on secp256k1)
Alice: private a (256-bit random), public A = a×G


Bob:   private b (256-bit random), public B = b×G


Shared secret point: True  ✓
Both compute a×b×G (same point, never transmitted)

────────────────────────────────────────────────────────────
ElGamal: security from α^a mod p  (needs ~3072-bit p)
ECC:     security from k×G         (needs ~256-bit N)
Same security, 12× smaller keys.


## 4.2 ECC Encryption / Decryption (ElGamal on Curves)

The essay demonstrates ECC encryption using the curve $y^2 = x^3 - x + 4$ over $\mathbb{F}_{457}$.
The scheme maps directly from ElGamal:

| ElGamal | ECC Analog |
|---------|------------|
| $\beta = \alpha^a \bmod p$ | $Q = d \times G$ (public key) |
| Encrypt: $(\alpha^k, m \cdot \beta^k)$ | Encrypt: $(k \times G, \; P_m + k \times Q)$ |
| Decrypt: $m = t \cdot (\beta')^{-a}$ | Decrypt: $P_m = C_2 - d \times C_1$ |

Multiplication in ElGamal becomes **point addition** in ECC.  
Exponentiation becomes **scalar multiplication**.

In [15]:
# ECC encryption/decryption on a small curve (from the essay)
# Curve: y² = x³ - x + 4 over F_457

P_SMALL = 457
A_SMALL = -1  # a coefficient
B_SMALL = 4   # b coefficient

class SmallPoint:
    def __init__(self, x=None, y=None):
        self.x = x
        self.y = y
    def is_infinity(self):
        return self.x is None
    def copy(self):
        return SmallPoint(self.x, self.y)
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity(): return True
        return self.x == other.x and self.y == other.y
    def __repr__(self):
        if self.is_infinity(): return "O"
        return f"({self.x}, {self.y})"

def small_add(p1, p2):
    if p1.is_infinity(): return p2.copy()
    if p2.is_infinity(): return p1.copy()
    if p1.x == p2.x:
        if (p1.y + p2.y) % P_SMALL == 0:
            return SmallPoint()
        lam = ((3 * p1.x * p1.x + A_SMALL) * pow(2 * p1.y, P_SMALL - 2, P_SMALL)) % P_SMALL
    else:
        lam = ((p2.y - p1.y) * pow(p2.x - p1.x, P_SMALL - 2, P_SMALL)) % P_SMALL
    x3 = (lam * lam - p1.x - p2.x) % P_SMALL
    y3 = (lam * (p1.x - x3) - p1.y) % P_SMALL
    return SmallPoint(x3, y3)

def small_mult(k, p):
    if k == 0 or p.is_infinity(): return SmallPoint()
    result = SmallPoint()
    addend = p.copy()
    while k > 0:
        if k & 1:
            result = small_add(result, addend)
        addend = small_add(addend, addend)
        k >>= 1
    return result

def small_negate(p):
    if p.is_infinity(): return SmallPoint()
    return SmallPoint(p.x, (P_SMALL - p.y) % P_SMALL)

def small_sqrt(a, p):
    """Square root mod p using Tonelli-Shanks (works for any odd prime)."""
    if a % p == 0:
        return 0
    if pow(a, (p - 1) // 2, p) != 1:
        return None
    if p % 4 == 3:
        return pow(a, (p + 1) // 4, p)
    q, s = p - 1, 0
    while q % 2 == 0:
        q //= 2
        s += 1
    z = 2
    while pow(z, (p - 1) // 2, p) != p - 1:
        z += 1
    m, c, t, r = s, pow(z, q, p), pow(a, q, p), pow(a, (q + 1) // 2, p)
    while t != 1:
        i = 1
        tmp = (t * t) % p
        while tmp != 1:
            tmp = (tmp * tmp) % p
            i += 1
        b = pow(c, 1 << (m - i - 1), p)
        m, c, t, r = i, (b * b) % p, (t * b * b) % p, (r * b) % p
    return r

# Setup: (4, 8) is on y² = x³ - x + 4 over F_457 since 8²=64 and 4³-4+4=64
G_small = SmallPoint(4, 8)
assert (8**2) % P_SMALL == (4**3 + A_SMALL*4 + B_SMALL) % P_SMALL, "G not on curve!"

# Key generation
d = 101  # Private key (small for demo)
Q = small_mult(d, G_small)  # Public key
print(f"=== ECC Encryption (Essay Example) ===")
print(f"Curve: y² = x³ - x + 4 over F_457")
print(f"G = {G_small}")
print(f"Private key d = {d}")
print(f"Public key Q = d×G = {Q}")

# Encoding: map message character to point (k=30 from essay)
K_ENC = 30

def encode_char_to_point(m_val):
    """Koblitz encoding: find point with x near K_ENC*m."""
    for j in range(K_ENC):
        x = K_ENC * m_val + j
        rhs = (x**3 + A_SMALL * x + B_SMALL) % P_SMALL
        y = small_sqrt(rhs, P_SMALL)
        if y is not None:
            return SmallPoint(x, y)
    return None

def decode_point_to_char(pt):
    return pt.x // K_ENC

# Encrypt 'H' (value = 7 in A=0..Z=25)
msg_val = 7  # 'H'
pm = encode_char_to_point(msg_val)
print(f"\nMessage: 'H' (value {msg_val}) → point {pm}")

# Encryption: (k×G, Pm + k×Q) with random k
k_rand = 41
C1 = small_mult(k_rand, G_small)       # k × G
C2 = small_add(pm, small_mult(k_rand, Q))  # Pm + k×Q
print(f"\nEncrypt with random k={k_rand}:")
print(f"  C1 = k×G = {C1}")
print(f"  C2 = Pm + k×Q = {C2}")

# Decryption: Pm = C2 - d×C1
dC1 = small_mult(d, C1)  # d × C1 = d×k×G = k×Q
pm_recovered = small_add(C2, small_negate(dC1))  # C2 - d×C1
msg_recovered = decode_point_to_char(pm_recovered)
print(f"\nDecrypt:")
print(f"  d×C1 = {dC1}")
print(f"  Pm = C2 - d×C1 = {pm_recovered}")
print(f"  Decoded: value {msg_recovered} → '{chr(65 + msg_recovered)}'")
print(f"  Match: {pm == pm_recovered}  ✓")

=== ECC Encryption (Essay Example) ===
Curve: y² = x³ - x + 4 over F_457
G = (4, 8)
Private key d = 101
Public key Q = d×G = (81, 387)

Message: 'H' (value 7) → point (210, 298)

Encrypt with random k=41:
  C1 = k×G = (445, 49)
  C2 = Pm + k×Q = (11, 180)

Decrypt:
  d×C1 = (427, 190)
  Pm = C2 - d×C1 = (210, 298)
  Decoded: value 7 → 'H'
  Match: True  ✓


### Why decryption works

$$C_2 - d \times C_1 = (P_m + k \times Q) - d \times (k \times G)$$
$$= P_m + k \times (d \times G) - d \times (k \times G)$$
$$= P_m + k \cdot d \times G - d \cdot k \times G$$
$$= P_m \quad \checkmark$$

The random factor $k$ cancels out because both parties have access to the
shared secret $k \cdot d \times G$ through different paths.

---

# Module 5: ECDSA — Digital Signatures

This is where everything we've built pays off.

In the intro we said the one-way function fires twice in Bitcoin. Now we have
the math to see exactly what that means:

- **Key generation** (happened once, when the wallet was created): you picked a
  secret integer $d$, computed $P = d \times G$, and published $P$ as your
  address. The one-way function is why nobody can reverse your address back to
  your private key — they'd have to solve the discrete log problem on secp256k1.

- **Signing** (happening right now, for this transaction): you pick a fresh
  random $k$, compute $R = k \times G$, and use $k$ together with $d$ and the
  message hash to produce the signature. The one-way function fires *again* —
  hiding $k$ inside $R$. If anyone could reverse $R$ back to $k$, they could
  solve for $d$ algebraically from the signing equation.

Both uses rely on the same hardness: given a point and the generator, you
cannot find the scalar. Everything below is the machinery that makes this work.

## 5.1 Signature Generation

Given private key $d$, message hash $z$:

1. Pick random nonce $k$
2. Compute $R = k \times G$ and let $r = R_x \bmod N$
3. Compute $s = k^{-1}(z + r \cdot d) \bmod N$
4. Signature is $(r, s)$

## 5.2 Signature Verification

Given public key $P$, message hash $z$, signature $(r, s)$:

1. Compute $u_1 = z \cdot s^{-1} \bmod N$
2. Compute $u_2 = r \cdot s^{-1} \bmod N$
3. Compute $R' = u_1 \times G + u_2 \times P$
4. Verify: $R'_x \bmod N \stackrel{?}{=} r$

In [16]:
import hashlib

def ecdsa_sign(message: bytes, private_key: int) -> tuple:
    """Generate ECDSA signature (r, s)."""
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    while True:
        k = secrets.randbelow(SECP_N - 1) + 1
        R = scalar_mult(k, G)
        r = R.x % SECP_N
        if r == 0:
            continue
        
        k_inv = pow(k, SECP_N - 2, SECP_N)
        s = (k_inv * (z + r * private_key)) % SECP_N
        if s == 0:
            continue
        
        return (r, s)

def ecdsa_verify(message: bytes, signature: tuple, public_key: Point) -> bool:
    """Verify ECDSA signature."""
    r, s = signature
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u1 = (z * s_inv) % SECP_N
    u2 = (r * s_inv) % SECP_N
    
    R_prime = point_add(scalar_mult(u1, G), scalar_mult(u2, public_key))
    
    return R_prime.x % SECP_N == r

# Generate key pair
d = secrets.randbelow(SECP_N - 1) + 1
P = scalar_mult(d, G)

# Sign a message
msg = b"Hello, Bitcoin!"
sig = ecdsa_sign(msg, d)
r, s = sig

print("=== ECDSA Demonstration ===")
print(f"Message: {msg.decode()}")
print(f"Private key d: {hex(d)[:20]}...")
print(f"Public key P:  {P}")
print(f"\nSignature:")
print(f"  r = {hex(r)[:20]}...")
print(f"  s = {hex(s)[:20]}...")

# Verify
valid = ecdsa_verify(msg, sig, P)
print(f"\nVerification: {valid}  ✓")

# Tamper with message
tampered = b"Hello, Bitcorn!"
valid_tampered = ecdsa_verify(tampered, sig, P)
print(f"Tampered msg verification: {valid_tampered}  ✗ (correctly rejected)")

# Wrong key
wrong_key = scalar_mult(secrets.randbelow(SECP_N - 1) + 1, G)
valid_wrong = ecdsa_verify(msg, sig, wrong_key)
print(f"Wrong key verification: {valid_wrong}  ✗ (correctly rejected)")

=== ECDSA Demonstration ===
Message: Hello, Bitcoin!
Private key d: 0x221aed6eb6ba68ac21...
Public key P:  (0x2ab8b502daee87..., 0x97aa0e2afe4699...)

Signature:
  r = 0x1343ea07f28d9218e1...
  s = 0xf3776d56fb8d91f497...



Verification: True  ✓


Tampered msg verification: False  ✗ (correctly rejected)


Wrong key verification: False  ✗ (correctly rejected)


## 5.3 Why Verification Works — The Proof

The verifier computes $R' = u_1 \times G + u_2 \times P$. Let's prove $R' = R$:

$$R' = u_1 G + u_2 P$$

Substitute $u_1 = z/s$, $u_2 = r/s$, and $P = dG$:

$$R' = \frac{z}{s} G + \frac{r}{s}(dG)$$

$$= \frac{z + rd}{s} G$$

From signing: $s = k^{-1}(z + rd)$, so $k = (z + rd)/s$:

$$R' = k G = R \quad \checkmark$$

## 5.4 The Nonce Catastrophe

If the same nonce $k$ is used for two different messages, the private key leaks:

$$s_1 = k^{-1}(z_1 + r \cdot d) \quad s_2 = k^{-1}(z_2 + r \cdot d)$$

$$s_1 - s_2 = k^{-1}(z_1 - z_2) \implies k = \frac{z_1 - z_2}{s_1 - s_2}$$

Once $k$ is known: $d = r^{-1}(sk - z)$. **Game over.**

This is exactly what happened to Sony's PS3 signing key in 2010.

---

In [17]:
# Demonstration: nonce reuse catastrophe

victim_priv = secrets.randbelow(SECP_N - 1) + 1
victim_pub = scalar_mult(victim_priv, G)

# Victim signs two messages with the SAME nonce (fatal mistake)
k_reused = secrets.randbelow(SECP_N - 1) + 1
R_k = scalar_mult(k_reused, G)
r_val = R_k.x % SECP_N
k_inv = pow(k_reused, SECP_N - 2, SECP_N)

msg1 = b"Transfer 1 BTC to Alice"
msg2 = b"Transfer 2 BTC to Bob"
z1 = int.from_bytes(hashlib.sha256(msg1).digest(), 'big')
z2 = int.from_bytes(hashlib.sha256(msg2).digest(), 'big')

s1 = (k_inv * (z1 + r_val * victim_priv)) % SECP_N
s2 = (k_inv * (z2 + r_val * victim_priv)) % SECP_N

print("=== Nonce Reuse Attack ===")
print(f"Attacker sees two signatures with same r:")
print(f"  sig1: r = {hex(r_val)[:16]}..., s1 = {hex(s1)[:16]}...")
print(f"  sig2: r = {hex(r_val)[:16]}..., s2 = {hex(s2)[:16]}...")
print(f"  Same r → same nonce k was used!")

# Attacker recovers k
k_recovered = ((z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N)) % SECP_N
print(f"\nAttacker recovers k: {k_recovered == k_reused}")

# Attacker recovers private key
d_recovered = (pow(r_val, SECP_N - 2, SECP_N) * (s1 * k_recovered - z1)) % SECP_N
print(f"Attacker recovers private key: {d_recovered == victim_priv}")
print(f"\n⚠️  NEVER reuse a nonce. Bitcoin uses RFC 6979 (deterministic k).")

=== Nonce Reuse Attack ===
Attacker sees two signatures with same r:
  sig1: r = 0xe69633691dabee..., s1 = 0x64301febb7af92...
  sig2: r = 0xe69633691dabee..., s2 = 0x867f8f8519ccee...
  Same r → same nonce k was used!

Attacker recovers k: True
Attacker recovers private key: True

⚠️  NEVER reuse a nonce. Bitcoin uses RFC 6979 (deterministic k).


---

# Module 6: Bitcoin Applications

Everything above converges in the real Bitcoin and Lightning Network protocols
implemented in this codebase.

## 6.1 From ECDSA to Schnorr (BIP340)

Bitcoin's Taproot upgrade (2021) moved from ECDSA to **Schnorr signatures**.

| Property | ECDSA | Schnorr (BIP340) |
|----------|-------|------------------|
| Signature size | ~72 bytes (DER) | 64 bytes (fixed) |
| Batch verification | No | Yes |
| Linearity | No | Yes (enables MuSig2) |
| Nonce generation | Random or RFC 6979 | BIP340 tagged hashes |

### Schnorr Signing

1. Nonce: $k$ → $R = k \times G$ (only x-coordinate used)
2. Challenge: $e = H(R_x \| P_x \| m)$
3. Response: $s = k + e \cdot d \pmod{N}$
4. Signature: $(R_x, s)$ — 64 bytes

### Schnorr Verification

$$s \times G \stackrel{?}{=} R + e \times P$$

Proof:
$$s \times G = (k + ed)G = kG + edG = R + eP \quad \checkmark$$

In [18]:
# Simplified Schnorr signature (BIP340-like)

def tagged_hash(tag: str, data: bytes) -> bytes:
    """BIP340 tagged hash: SHA256(SHA256(tag) || SHA256(tag) || data)."""
    tag_hash = hashlib.sha256(tag.encode()).digest()
    return hashlib.sha256(tag_hash + tag_hash + data).digest()

def schnorr_sign(message: bytes, private_key: int) -> bytes:
    """Simplified BIP340 Schnorr signature."""
    P = scalar_mult(private_key, G)
    d = private_key if P.y % 2 == 0 else SECP_N - private_key
    
    # Deterministic nonce (simplified)
    aux = secrets.token_bytes(32)
    t = int.from_bytes(aux, 'big') ^ d
    k_bytes = tagged_hash("BIP0340/nonce", 
                          t.to_bytes(32, 'big') + P.x.to_bytes(32, 'big') + message)
    k = int.from_bytes(k_bytes, 'big') % SECP_N
    if k == 0:
        raise ValueError("k is zero")
    
    R = scalar_mult(k, G)
    if R.y % 2 != 0:
        k = SECP_N - k
        R = scalar_mult(k, G)
    
    e_bytes = tagged_hash("BIP0340/challenge",
                          R.x.to_bytes(32, 'big') + P.x.to_bytes(32, 'big') + message)
    e = int.from_bytes(e_bytes, 'big') % SECP_N
    
    s = (k + e * d) % SECP_N
    return R.x.to_bytes(32, 'big') + s.to_bytes(32, 'big')

def schnorr_verify(message: bytes, signature: bytes, pubkey_x: int) -> bool:
    """Simplified BIP340 Schnorr verification."""
    R_x = int.from_bytes(signature[:32], 'big')
    s = int.from_bytes(signature[32:], 'big')
    
    # Recover P (assume even y)
    y2 = (pow(pubkey_x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if y % 2 != 0:
        y = SECP_P - y
    P = Point(pubkey_x, y)
    
    e_bytes = tagged_hash("BIP0340/challenge",
                          R_x.to_bytes(32, 'big') + pubkey_x.to_bytes(32, 'big') + message)
    e = int.from_bytes(e_bytes, 'big') % SECP_N
    
    # Verify: s×G = R + e×P
    lhs = scalar_mult(s, G)
    
    # Recover R (assume even y)
    R_y2 = (pow(R_x, 3, SECP_P) + 7) % SECP_P
    R_y = mod_sqrt(R_y2, SECP_P)
    if R_y % 2 != 0:
        R_y = SECP_P - R_y
    R = Point(R_x, R_y)
    
    rhs = point_add(R, scalar_mult(e, P))
    return lhs == rhs

# Demo
d_schnorr = secrets.randbelow(SECP_N - 1) + 1
P_schnorr = scalar_mult(d_schnorr, G)
msg_schnorr = b"Taproot transaction"

sig_schnorr = schnorr_sign(msg_schnorr, d_schnorr)
valid_schnorr = schnorr_verify(msg_schnorr, sig_schnorr, P_schnorr.x)

print("=== Schnorr Signature (BIP340) ===")
print(f"Message: {msg_schnorr.decode()}")
print(f"Signature: {sig_schnorr.hex()[:40]}...")
print(f"  R_x (32 bytes) + s (32 bytes) = {len(sig_schnorr)} bytes total")
print(f"Verification: {valid_schnorr}  ✓")
print(f"\nCompare: ECDSA ≈72 bytes (DER),  Schnorr = 64 bytes (fixed)")

=== Schnorr Signature (BIP340) ===
Message: Taproot transaction
Signature: 314f341f474c4f2f024e2d264571b9bec78890a4...
  R_x (32 bytes) + s (32 bytes) = 64 bytes total
Verification: True  ✓

Compare: ECDSA ≈72 bytes (DER),  Schnorr = 64 bytes (fixed)


## 6.2 ECDH in Lightning Onion Routing

Lightning's **onion routing** uses ECC for privacy:

```
Sender generates ephemeral key pair: (e, E = e×G)

For each hop node with public key P_i:
  1. Shared secret: S_i = SHA256(e × P_i)    ← ECDH
  2. Derive keys: rho_i = HMAC("rho", S_i)    ← encryption
                  mu_i  = HMAC("mu", S_i)     ← authentication
  3. Encrypt routing info with ChaCha20(rho_i)
  4. Blind ephemeral key: E' = E × SHA256(E || S_i)
                                ↑ prevents linking between hops
```

Each hop can compute the same shared secret using its private key:
$$S_i = \text{SHA256}(d_i \times E)$$

This works because $d_i \times E = d_i \times (e \times G) = e \times (d_i \times G) = e \times P_i$

In [19]:
import hmac as hmac_lib

def ecdh_shared_secret(my_private: int, their_public: Point) -> bytes:
    """ECDH: compute shared secret from private key and other party's public key."""
    shared_point = scalar_mult(my_private, their_public)
    compressed = serialize_compressed(shared_point)
    return hashlib.sha256(compressed).digest()

def generate_key(shared_secret: bytes, key_type: str) -> bytes:
    """Derive a specific key from shared secret (BOLT #4 key derivation)."""
    return hmac_lib.new(key_type.encode(), shared_secret, hashlib.sha256).digest()

def blind_ephemeral(ephemeral_pub: Point, shared_secret: bytes) -> Point:
    """Blind ephemeral key so next hop can't link it to previous hop."""
    pub_bytes = serialize_compressed(ephemeral_pub)
    blind_bytes = hashlib.sha256(pub_bytes + shared_secret).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    return scalar_mult(blind_factor, ephemeral_pub)

# Simulate 3-hop onion routing
print("=== Lightning Onion Routing (ECDH) ===")
print("Sender → Hop1 → Hop2 → Hop3 (recipient)")

# Each hop has a key pair
hops = []
for i in range(3):
    d_hop = secrets.randbelow(SECP_N - 1) + 1
    P_hop = scalar_mult(d_hop, G)
    hops.append({'private': d_hop, 'public': P_hop, 'name': f'Hop{i+1}'})

# Sender creates ephemeral key pair
e_priv = secrets.randbelow(SECP_N - 1) + 1
E = scalar_mult(e_priv, G)  # Ephemeral public key (sent with packet)
print(f"\nSender ephemeral E: {serialize_compressed(E).hex()[:20]}...")

# Sender computes all shared secrets
ephemeral = E
sender_secrets = []
for hop in hops:
    ss = ecdh_shared_secret(e_priv, hop['public'])
    rho = generate_key(ss, "rho")
    mu = generate_key(ss, "mu")
    sender_secrets.append({'ss': ss, 'rho': rho, 'mu': mu})
    
    # Blind for next hop
    blind_bytes = hashlib.sha256(serialize_compressed(ephemeral) + ss).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    e_priv = (e_priv * blind_factor) % SECP_N
    ephemeral = scalar_mult(e_priv, G)

# Each hop computes the same shared secret using its private key
E_current = E
for i, hop in enumerate(hops):
    hop_ss = ecdh_shared_secret(hop['private'], E_current)
    match = hop_ss == sender_secrets[i]['ss']
    print(f"\n{hop['name']}:")
    print(f"  Receives E: {serialize_compressed(E_current).hex()[:20]}...")
    print(f"  Computes: d×E = shared secret")
    print(f"  Shared secret matches sender's: {match}  ✓")
    print(f"  Derives: rho (encrypt), mu (HMAC)")
    
    # Blind E for next hop
    blind_bytes = hashlib.sha256(serialize_compressed(E_current) + hop_ss).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    E_current = scalar_mult(blind_factor, E_current)

print(f"\nKey insight: each hop sees a different E (blinding prevents linking).")
print(f"Nobody except the sender knows the full route.")

=== Lightning Onion Routing (ECDH) ===
Sender → Hop1 → Hop2 → Hop3 (recipient)



Sender ephemeral E: 02250c136c4923ed4d90...



Hop1:
  Receives E: 02250c136c4923ed4d90...
  Computes: d×E = shared secret
  Shared secret matches sender's: True  ✓
  Derives: rho (encrypt), mu (HMAC)

Hop2:
  Receives E: 0221188c8feaec410e41...
  Computes: d×E = shared secret
  Shared secret matches sender's: True  ✓
  Derives: rho (encrypt), mu (HMAC)



Hop3:
  Receives E: 02d56b7cba19a850dceb...
  Computes: d×E = shared secret
  Shared secret matches sender's: True  ✓
  Derives: rho (encrypt), mu (HMAC)

Key insight: each hop sees a different E (blinding prevents linking).
Nobody except the sender knows the full route.


---

# Module 7: Self-Assessment & Exercises

## 7.1 Concept Check

Answer each question, then run the verification cell below.

---

In [20]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 1: Finite Field Arithmetic
# ═══════════════════════════════════════════════════════════════
#
#  Compute the multiplicative inverse of 7 in F_11.
#  That is, find x such that 7 * x ≡ 1 (mod 11).

answer_1 = 0  # <-- Replace with your answer

# Verify
if (7 * answer_1) % 11 == 1:
    print(f"Exercise 1: ✓  7 × {answer_1} = {7 * answer_1} ≡ {(7*answer_1)%11} (mod 11)")
else:
    print(f"Exercise 1: ✗  7 × {answer_1} = {7 * answer_1} ≡ {(7*answer_1)%11} (mod 11), need ≡ 1")
    print(f"  Hint: use Fermat's little theorem: 7^(11-2) mod 11")

Exercise 1: ✗  7 × 0 = 0 ≡ 0 (mod 11), need ≡ 1
  Hint: use Fermat's little theorem: 7^(11-2) mod 11


In [21]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 2: Points on a Curve
# ═══════════════════════════════════════════════════════════════
#
#  For the curve y² = x³ + 7 over F_11, find a valid point.
#  Try x values and check if x³ + 7 is a quadratic residue mod 11.
#  If it is, compute y.

x_answer = 0  # <-- Replace with x
y_answer = 0  # <-- Replace with y

# Verify
p_ex = 11
lhs_ex = (y_answer ** 2) % p_ex
rhs_ex = (x_answer ** 3 + 7) % p_ex
if lhs_ex == rhs_ex and not (x_answer == 0 and y_answer == 0):
    print(f"Exercise 2: ✓  ({x_answer}, {y_answer}) is on y² = x³ + 7 over F_11")
    print(f"  y² = {y_answer}² = {y_answer**2} ≡ {lhs_ex} (mod 11)")
    print(f"  x³+7 = {x_answer}³+7 = {x_answer**3+7} ≡ {rhs_ex} (mod 11)")
else:
    print(f"Exercise 2: ✗  ({x_answer}, {y_answer}) is NOT on the curve")
    print(f"  y² mod 11 = {lhs_ex}, x³+7 mod 11 = {rhs_ex}")
    print(f"  Hint: try x=2. What is 2³ + 7 = {2**3 + 7} mod 11 = {(2**3 + 7) % 11}?")
    print(f"  Is {(2**3 + 7) % 11} a perfect square mod 11?")

Exercise 2: ✗  (0, 0) is NOT on the curve
  y² mod 11 = 0, x³+7 mod 11 = 7
  Hint: try x=2. What is 2³ + 7 = 15 mod 11 = 4?
  Is 4 a perfect square mod 11?


In [22]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 3: Point Negation
# ═══════════════════════════════════════════════════════════════
#
#  If P = (3, 10) on curve y² = x³ + 7 over F_11,
#  what is -P?

neg_x = 0  # <-- Replace
neg_y = 0  # <-- Replace

# Verify
expected_neg_y = (11 - 10) % 11
if neg_x == 3 and neg_y == expected_neg_y:
    print(f"Exercise 3: ✓  -P = ({neg_x}, {neg_y})")
    print(f"  Negation flips y: -(3, 10) = (3, 11-10) = (3, {expected_neg_y})")
else:
    print(f"Exercise 3: ✗  Got ({neg_x}, {neg_y})")
    print(f"  Hint: negation = (x, P-y) = (3, 11-10) = ?")

Exercise 3: ✗  Got (0, 0)
  Hint: negation = (x, P-y) = (3, 11-10) = ?


In [23]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 4: Double-and-Add Trace
# ═══════════════════════════════════════════════════════════════
#
#  How many point doublings and point additions does it take
#  to compute 45 × G using double-and-add?
#
#  Steps:
#  1. Write 45 in binary: ____________
#  2. Count the bits (= number of doublings)
#  3. Count the 1-bits (= number of additions)

binary_45 = ""     # <-- Write 45 in binary (e.g., "110")
num_doubles = 0     # <-- How many doublings?
num_adds = 0        # <-- How many additions?

# Verify
actual_binary = bin(45)[2:]
actual_doubles = len(actual_binary) - 1  # Don't double on last bit
actual_adds = actual_binary.count('1')

if binary_45 == actual_binary:
    print(f"Exercise 4a: ✓  45 in binary = {actual_binary}")
else:
    print(f"Exercise 4a: ✗  45 in binary = {actual_binary}, you wrote '{binary_45}'")

if num_doubles == actual_doubles:
    print(f"Exercise 4b: ✓  {actual_doubles} doublings")
else:
    print(f"Exercise 4b: ✗  Expected {actual_doubles} doublings, got {num_doubles}")

if num_adds == actual_adds:
    print(f"Exercise 4c: ✓  {actual_adds} additions")
else:
    print(f"Exercise 4c: ✗  Expected {actual_adds} additions, got {num_adds}")

print(f"\nNaive would need 44 additions. Double-and-add: {actual_doubles + actual_adds} operations.")

Exercise 4a: ✗  45 in binary = 101101, you wrote ''
Exercise 4b: ✗  Expected 5 doublings, got 0
Exercise 4c: ✗  Expected 4 additions, got 0

Naive would need 44 additions. Double-and-add: 9 operations.


In [24]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 5: ECDSA — Verify by Hand
# ═══════════════════════════════════════════════════════════════
#
#  Given the ECDSA verification equation:
#    R' = u1×G + u2×P
#  where u1 = z/s mod N and u2 = r/s mod N
#
#  Fill in the proof that R' = R (the original nonce point).
#  
#  Starting from: R' = (z/s)×G + (r/s)×P
#  Substitute P = d×G:
#    R' = (z/s)×G + (r/s)×(d×G)
#       = _____________ × G        ← factor out G
#  Since s = k⁻¹(z + r×d):
#    (z + r×d)/s = (z + r×d) / (k⁻¹(z + r×d)) = _____
#  Therefore R' = _____ × G = R    QED

factored = ""       # <-- What's the scalar before ×G? (e.g., "(z+rd)/s")
simplified = ""     # <-- What does (z+rd)/s simplify to?

correct_factored = "(z+rd)/s"
correct_simplified = "k"

if factored.replace(" ", "") == correct_factored:
    print(f"Exercise 5a: ✓  R' = {factored} × G")
else:
    print(f"Exercise 5a: ✗  R' = {correct_factored} × G")

if simplified.lower().strip() == correct_simplified:
    print(f"Exercise 5b: ✓  Simplifies to {simplified}, so R' = k×G = R  ✓")
else:
    print(f"Exercise 5b: ✗  (z+rd)/s simplifies to k (because s = (z+rd)/k)")

Exercise 5a: ✗  R' = (z+rd)/s × G
Exercise 5b: ✗  (z+rd)/s simplifies to k (because s = (z+rd)/k)


In [25]:
# ═══════════════════════════════════════════════════════════════
#  EXERCISE 6: Implement Nonce Recovery
# ═══════════════════════════════════════════════════════════════
#
#  Given two signatures (r, s1) and (r, s2) for messages with
#  hashes z1 and z2 (same r means same nonce!), recover the
#  nonce k and then the private key d.
#
#  Formulas:
#    k = (z1 - z2) × (s1 - s2)⁻¹ mod N
#    d = r⁻¹ × (s1×k - z1) mod N

# Setup (don't modify)
ex6_d = secrets.randbelow(SECP_N - 1) + 1
ex6_k = secrets.randbelow(SECP_N - 1) + 1
ex6_R = scalar_mult(ex6_k, G)
ex6_r = ex6_R.x % SECP_N
ex6_k_inv = pow(ex6_k, SECP_N - 2, SECP_N)

ex6_z1 = int.from_bytes(hashlib.sha256(b"message one").digest(), 'big')
ex6_z2 = int.from_bytes(hashlib.sha256(b"message two").digest(), 'big')
ex6_s1 = (ex6_k_inv * (ex6_z1 + ex6_r * ex6_d)) % SECP_N
ex6_s2 = (ex6_k_inv * (ex6_z2 + ex6_r * ex6_d)) % SECP_N

# YOUR CODE: recover k and d
recovered_k = 0  # <-- Replace with formula
recovered_d = 0  # <-- Replace with formula

# Verify
if recovered_k == ex6_k:
    print(f"Exercise 6a: ✓  Nonce k recovered!")
else:
    print(f"Exercise 6a: ✗  k not recovered")
    print(f"  Hint: k = (z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N) % SECP_N")

if recovered_d == ex6_d:
    print(f"Exercise 6b: ✓  Private key d recovered! (This is why nonce reuse is fatal)")
else:
    print(f"Exercise 6b: ✗  d not recovered")
    print(f"  Hint: d = pow(r, SECP_N - 2, SECP_N) * (s1 * k - z1) % SECP_N")

Exercise 6a: ✗  k not recovered
  Hint: k = (z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N) % SECP_N
Exercise 6b: ✗  d not recovered
  Hint: d = pow(r, SECP_N - 2, SECP_N) * (s1 * k - z1) % SECP_N


## 7.2 Knowledge Map

Check off each concept as you understand it:

```
Module 1: Foundations                    Module 4: ElGamal vs ECC
[ ] Abelian group axioms                [ ] Discrete log in both settings
[ ] Finite field F_p                    [ ] Key size comparison
[ ] Modular inverse via Fermat          [ ] ECDH key exchange
[ ] Equivalence classes mod n           [ ] ECC encryption/decryption
[ ] Projective coordinates
                                        Module 5: ECDSA
Module 2: Elliptic Curves               [ ] Signature generation
[ ] Weierstrass equation                [ ] Signature verification
[ ] secp256k1 parameters                [ ] The nonce catastrophe
[ ] Curve over finite field             [ ] Recoverable signatures
[ ] Non-singularity condition
                                        Module 6: Bitcoin
Module 3: Point Operations              [ ] Schnorr vs ECDSA
[ ] Point addition formula              [ ] ECDH in onion routing
[ ] Point doubling formula              [ ] Ephemeral key blinding
[ ] Point at infinity (identity)
[ ] Scalar multiplication
[ ] Double-and-add algorithm
[ ] Compressed public keys
```

---

## 7.3 Further Study

### External references:

- **Original essay source:** `esixce/ecdsa-lab` on GitHub
- [SEC 2: Recommended Elliptic Curve Domain Parameters](https://www.secg.org/sec2-v2.pdf) (secp256k1 spec)
- [BIP340: Schnorr Signatures](https://github.com/bitcoin/bips/blob/master/bip-0340.mediawiki)
- [BOLT #4: Onion Routing Protocol](https://github.com/lightning/bolts/blob/master/04-onion-routing.md)
- Johnson, Menezes, Vanstone: *The Elliptic Curve Digital Signature Algorithm (ECDSA)*, Certicom 2001

### Security considerations:

All implementations in this notebook are **educational**. Production code must:
- Use constant-time operations (timing attacks)
- Use battle-tested libraries (`libsecp256k1`, `ring`, `openssl`)
- Validate all inputs (point-on-curve, subgroup order)
- Never reuse nonces
- Zeroize secrets after use
